# Accepted Loan EDA: LendingClub 2007-2018Q4

## Executive Summary

This notebook performs a rigorous exploratory data analysis of **accepted LendingClub loans only**. It intentionally does **not** load, join, profile, or infer from denied/rejected loan applications. The modeling objective is repayment risk conditional on the loan already being accepted and originated.

**Scope decision:** this is an accepted-population repayment-risk analysis, not an application-decision or reject-inference study. As a result, conclusions are valid for originated LendingClub loans and should not be generalized to all applicants without a separate selection-bias/reject-inference design.

**Core EDA questions:**

1. What is the accepted-loan portfolio composition by time, grade, term, purpose, geography, and borrower attributes?
2. Which fields are clean enough for modeling, which are sparse, and which require imputation or exclusion?
3. How should repayment outcome be defined using only observed accepted-loan performance?
4. How do observed bad rates vary across underwriting-relevant variables such as grade, sub-grade, FICO, DTI, term, income, and verification status?
5. Which accepted-loan fields are clear post-origination leakage and must be excluded from model training?
6. What modeling population, features, and validation split are defensible for credit-risk modeling?


## Notebook Design

The notebook is organized for a credit-risk review workflow:

- Setup and accepted-file path validation
- Memory-aware schema and loading strategy
- Cleaning and type conversion
- Target definition for accepted loans
- Missingness and data-quality diagnostics
- Univariate and bivariate EDA
- Temporal/vintage EDA
- Outcome and observed bad-rate EDA
- Leakage audit and modeling-readiness recommendations

Important conventions:

- **Only accepted loans are analyzed.** No rejected/denied file is read or referenced.
- Raw source columns are preserved where practical; cleaned helper columns are added.
- Sampling is used for plots and local EDA. Exact scans are exposed as explicit switches because the file has more than 2.2M rows.
- Bad-rate estimates are shown only for groups with sufficient observations to reduce noise.


In [ ]:
from __future__ import annotations

import gc
import gzip
import os
import platform
import re
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Iterable

# Keep matplotlib cache in a writable temp location when the default home cache is unavailable.
os.environ.setdefault("MPLCONFIGDIR", "/private/tmp/lendingclub_mplconfig")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)

pd.set_option("display.max_columns", 180)
pd.set_option("display.max_rows", 140)
pd.set_option("display.width", 200)
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
PLOT_DPI = 150
rng = np.random.default_rng(RANDOM_STATE)

print("Python:", platform.python_version())
print("pandas:", pd.__version__)
print("numpy:", np.__version__)


## 1. Configuration and Accepted-File Validation

This notebook uses the same project and data-root convention as `LendingClub_EDA`:

- `PROJECT_ROOT` resolves to `Final/CreditRiskRAG` through `.env`, local project markers, or the same absolute fallback path.
- `DATA_ROOT` is `PROJECT_ROOT.parent / "Data" / "archive"`.
- Accepted-loan discovery uses `accepted_*.csv*` and prefers the compressed `.csv.gz` file when present.

The rejected file is intentionally absent from this configuration.


In [ ]:
DEFAULT_PROJECT_ROOT = Path(
    "/Users/lindaperez/Documents/NEU/2026SummerML/MachineLearningClass/"
    "Final_Project/Final/CreditRiskRAG"
)


def load_env_file(env_path: Path) -> None:
    """Load simple KEY=VALUE pairs without overriding existing environment variables."""
    if not env_path.exists():
        return

    for raw_line in env_path.read_text().splitlines():
        line = raw_line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, value = line.split("=", 1)
        os.environ.setdefault(key.strip(), value.strip().strip('"').strip("'"))


def find_project_root() -> Path:
    """Match LendingClub_EDA project-root resolution."""
    cwd = Path.cwd().resolve()

    for candidate in (cwd, *cwd.parents):
        load_env_file(candidate / ".env")
        env_root = os.environ.get("PROJECT_ROOT")
        if env_root:
            return Path(env_root).expanduser().resolve()

        looks_like_project = (
            (candidate / "EDA").exists()
            and (candidate / "README.md").exists()
        )
        if looks_like_project:
            return candidate

    return DEFAULT_PROJECT_ROOT


PROJECT_ROOT = find_project_root()
DATA_ROOT = PROJECT_ROOT.parent / "Data" / "archive"


@dataclass(frozen=True)
class AcceptedEDAConfig:
    project_root: Path = PROJECT_ROOT
    data_dir: Path = DATA_ROOT
    accepted_pattern: str = "accepted_*.csv*"
    sample_rows: int = 250_000
    chunk_size: int = 250_000
    min_group_n: int = 500
    output_dir: Path = PROJECT_ROOT / "EDA" / "accepted_eda_outputs"


def discover_table_file(data_dir: Path, pattern: str) -> Path | None:
    """Match LendingClub_EDA file discovery and prefer compressed .csv.gz files."""
    candidates = sorted(path for path in data_dir.glob(pattern) if path.is_file())
    if not candidates:
        return None
    return sorted(
        candidates,
        key=lambda path: (
            not path.name.lower().endswith(".csv.gz"),
            len(path.name),
        ),
    )[0]


CONFIG = AcceptedEDAConfig()
ACCEPTED_PATH = discover_table_file(CONFIG.data_dir, CONFIG.accepted_pattern)

CONFIG.output_dir.mkdir(parents=True, exist_ok=True)
PLOT_DIR = CONFIG.output_dir / "plots"
TABLE_DIR = CONFIG.output_dir / "tables"
PLOT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

if ACCEPTED_PATH is None or not ACCEPTED_PATH.exists():
    raise FileNotFoundError(
        f"Missing accepted LendingClub data matching {CONFIG.accepted_pattern} "
        f"in {CONFIG.data_dir}"
    )

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("Accepted loan source:", ACCEPTED_PATH)
print("Exists:", ACCEPTED_PATH.exists())
print("Size MB:", round(ACCEPTED_PATH.stat().st_size / 1024**2, 2))


## 2. Reusable EDA Utilities

These helpers provide file metadata, header reads, deterministic sampling, plot saving, categorical summaries, and bad-rate summaries. They are written to avoid loading the full raw file unless explicitly requested.


In [ ]:
def is_gzip_path(path: Path) -> bool:
    return "".join(path.suffixes).lower().endswith(".gz")


def read_header(path: Path) -> list[str]:
    opener = gzip.open if is_gzip_path(path) else open
    mode = "rt" if is_gzip_path(path) else "r"
    with opener(path, mode, newline="", errors="replace") as f:
        return f.readline().rstrip("\n").split(",")


def slugify(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")


def save_current_plot(name: str) -> Path:
    path = PLOT_DIR / f"{slugify(name)}.png"
    plt.tight_layout()
    plt.savefig(path, dpi=PLOT_DPI, bbox_inches="tight")
    return path


def count_rows_csv(path: Path, chunk_size: int = CONFIG.chunk_size, usecols: list[str] | None = None) -> int:
    total = 0
    for chunk in pd.read_csv(path, chunksize=chunk_size, usecols=usecols, low_memory=False):
        total += len(chunk)
    return total


def sample_csv_chunks(
    path: Path,
    target_rows: int,
    chunk_size: int,
    usecols: list[str] | None = None,
    random_state: int = RANDOM_STATE,
) -> pd.DataFrame:
    samples = []
    total_seen = 0
    rng_local = np.random.default_rng(random_state)

    for chunk in pd.read_csv(path, usecols=usecols, chunksize=chunk_size, low_memory=False):
        total_seen += len(chunk)
        keep_prob = min(1.0, target_rows / max(total_seen, 1))
        if keep_prob >= 1.0:
            sampled = chunk
        else:
            mask = rng_local.random(len(chunk)) < keep_prob
            sampled = chunk.loc[mask]
        if len(sampled):
            samples.append(sampled)
        current = sum(len(x) for x in samples)
        if current >= target_rows * 1.5:
            break

    if not samples:
        return pd.DataFrame()

    out = pd.concat(samples, ignore_index=True)
    if len(out) > target_rows:
        out = out.sample(n=target_rows, random_state=random_state)
    return out.reset_index(drop=True)


def display_and_save_table(df: pd.DataFrame, name: str, index: bool = False) -> pd.DataFrame:
    path = TABLE_DIR / f"{slugify(name)}.csv"
    df.to_csv(path, index=index)
    display(df)
    print("Saved:", path)
    return df


def categorical_summary(df: pd.DataFrame, col: str, top_n: int = 20) -> pd.DataFrame:
    counts = df[col].value_counts(dropna=False).head(top_n)
    return pd.DataFrame({"count": counts, "pct": (counts / len(df) * 100).round(2)})


def bad_rate_by_group(
    df: pd.DataFrame,
    group_col: str,
    target_col: str = "target_bad",
    min_n: int = CONFIG.min_group_n,
) -> pd.DataFrame:
    tmp = df[df[target_col].notna() & df[group_col].notna()].copy()
    if tmp.empty:
        return pd.DataFrame(columns=[group_col, "n", "bad_rate", "bad_count"])
    out = (
        tmp.groupby(group_col, observed=True)[target_col]
        .agg(n="count", bad_rate="mean", bad_count="sum")
        .reset_index()
    )
    out = out[out["n"] >= min_n].sort_values("bad_rate", ascending=False)
    out["bad_rate"] = out["bad_rate"].round(4)
    out["bad_count"] = out["bad_count"].astype(int)
    return out


## 3. File and Schema Overview

This section inventories the accepted-loan file, raw columns, and a small leading sample before any cleaning. It establishes the available schema and confirms the presence of key credit-risk and outcome columns.


In [ ]:
accepted_columns = read_header(ACCEPTED_PATH)
print(f"Accepted raw columns: {len(accepted_columns)}")
print(accepted_columns)

RUN_EXACT_ROW_COUNT = True
if RUN_EXACT_ROW_COUNT:
    accepted_row_count = count_rows_csv(ACCEPTED_PATH, usecols=["id"] if "id" in accepted_columns else None)
else:
    accepted_row_count = None

file_overview = pd.DataFrame([
    {
        "dataset": "accepted_loans",
        "path": str(ACCEPTED_PATH),
        "size_mb": round(ACCEPTED_PATH.stat().st_size / 1024**2, 2),
        "columns": len(accepted_columns),
        "rows_exact": accepted_row_count,
    }
])
display_and_save_table(file_overview, "accepted_file_overview")


In [ ]:
accepted_head = pd.read_csv(ACCEPTED_PATH, nrows=5_000, low_memory=False)

schema_summary = pd.DataFrame({
    "column": accepted_head.columns,
    "dtype_sample": accepted_head.dtypes.astype(str).values,
    "non_null_sample": accepted_head.notna().sum().values,
    "missing_pct_sample": accepted_head.isna().mean().mul(100).round(2).values,
    "unique_sample": accepted_head.nunique(dropna=True).values,
    "example": [accepted_head[c].dropna().iloc[0] if accepted_head[c].notna().any() else np.nan for c in accepted_head.columns],
}).sort_values(["missing_pct_sample", "column"], ascending=[False, True])

display_and_save_table(schema_summary, "accepted_schema_sample")


## 4. Memory-Aware Working Sample

The working EDA sample keeps application-time credit attributes, selected origination attributes, the performance outcome, and a small number of servicing fields only for leakage demonstration. Servicing/post-outcome fields are not proposed as model features.


In [ ]:
APPLICATION_TIME_CANDIDATES = [
    "loan_amnt", "term", "int_rate", "installment", "grade", "sub_grade",
    "emp_length", "home_ownership", "annual_inc", "verification_status",
    "issue_d", "purpose", "zip_code", "addr_state", "dti", "delinq_2yrs",
    "earliest_cr_line", "fico_range_low", "fico_range_high", "inq_last_6mths",
    "mths_since_last_delinq", "mths_since_last_record", "open_acc", "pub_rec",
    "revol_bal", "revol_util", "total_acc", "initial_list_status", "collections_12_mths_ex_med",
    "mths_since_last_major_derog", "policy_code", "application_type", "annual_inc_joint",
    "dti_joint", "verification_status_joint", "acc_now_delinq", "tot_coll_amt", "tot_cur_bal",
    "open_acc_6m", "open_act_il", "open_il_12m", "open_il_24m", "mths_since_rcnt_il",
    "total_bal_il", "il_util", "open_rv_12m", "open_rv_24m", "max_bal_bc", "all_util",
    "total_rev_hi_lim", "inq_fi", "total_cu_tl", "inq_last_12m", "acc_open_past_24mths",
    "avg_cur_bal", "bc_open_to_buy", "bc_util", "chargeoff_within_12_mths", "delinq_amnt",
    "mo_sin_old_il_acct", "mo_sin_old_rev_tl_op", "mo_sin_rcnt_rev_tl_op", "mo_sin_rcnt_tl",
    "mort_acc", "mths_since_recent_bc", "mths_since_recent_bc_dlq", "mths_since_recent_inq",
    "mths_since_recent_revol_delinq", "num_accts_ever_120_pd", "num_actv_bc_tl",
    "num_actv_rev_tl", "num_bc_sats", "num_bc_tl", "num_il_tl", "num_op_rev_tl",
    "num_rev_accts", "num_rev_tl_bal_gt_0", "num_sats", "num_tl_120dpd_2m", "num_tl_30dpd",
    "num_tl_90g_dpd_24m", "num_tl_op_past_12m", "pct_tl_nvr_dlq", "percent_bc_gt_75",
    "pub_rec_bankruptcies", "tax_liens", "tot_hi_cred_lim", "total_bal_ex_mort", "total_bc_limit",
    "total_il_high_credit_limit", "disbursement_method",
]

IDENTIFIER_COLS = ["id", "member_id", "url"]
TARGET_COLS = ["loan_status"]
LEAKAGE_DEMO_COLS = [
    "out_prncp", "total_pymnt", "total_rec_prncp", "total_rec_int", "recoveries",
    "collection_recovery_fee", "last_pymnt_d", "last_pymnt_amnt", "last_credit_pull_d",
    "last_fico_range_high", "last_fico_range_low", "debt_settlement_flag", "hardship_flag",
]

requested_cols = IDENTIFIER_COLS + TARGET_COLS + APPLICATION_TIME_CANDIDATES + LEAKAGE_DEMO_COLS
ACCEPTED_USECOLS = [c for c in requested_cols if c in accepted_columns]
missing_requested = sorted(set(requested_cols) - set(ACCEPTED_USECOLS))
print(f"Columns requested for working EDA: {len(ACCEPTED_USECOLS)}")
print(f"Requested columns unavailable in this file: {missing_requested}")

accepted_raw = sample_csv_chunks(
    ACCEPTED_PATH,
    target_rows=CONFIG.sample_rows,
    chunk_size=CONFIG.chunk_size,
    usecols=ACCEPTED_USECOLS,
    random_state=RANDOM_STATE,
)
print("Working sample shape:", accepted_raw.shape)
display(accepted_raw.head())


## 5. Cleaning and Type Conversion

The cleaning step standardizes dates, terms, employment length, percentages, FICO, and common numeric fields. Raw files are unchanged.


In [ ]:
def parse_percent(series: pd.Series) -> pd.Series:
    if pd.api.types.is_numeric_dtype(series):
        return pd.to_numeric(series, errors="coerce")
    cleaned = series.astype("string").str.replace("%", "", regex=False).str.strip()
    return pd.to_numeric(cleaned, errors="coerce")


def parse_term_months(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series.astype("string").str.extract(r"(\d+)")[0], errors="coerce").astype("Int16")


def parse_emp_length(series: pd.Series) -> pd.Series:
    s = series.astype("string").str.lower().str.strip()
    out = pd.Series(np.nan, index=series.index, dtype="float")
    out[s.str.contains("< 1", na=False)] = 0
    out[s.str.contains(r"10\+", na=False)] = 10
    extracted = pd.to_numeric(s.str.extract(r"(\d+)")[0], errors="coerce")
    out = out.fillna(extracted)
    return out


def parse_lc_month(series: pd.Series) -> pd.Series:
    return pd.to_datetime(series, format="%b-%Y", errors="coerce")


def add_clean_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    for col in ["int_rate", "revol_util"]:
        if col in out:
            out[f"{col}_clean"] = parse_percent(out[col])

    if "term" in out:
        out["term_months"] = parse_term_months(out["term"])
    if "emp_length" in out:
        out["emp_length_years"] = parse_emp_length(out["emp_length"])
    if {"fico_range_low", "fico_range_high"}.issubset(out.columns):
        out["fico_mean"] = out[["fico_range_low", "fico_range_high"]].mean(axis=1)

    for col in ["issue_d", "earliest_cr_line", "last_pymnt_d", "last_credit_pull_d"]:
        if col in out:
            out[f"{col}_dt"] = parse_lc_month(out[col])

    if "issue_d_dt" in out:
        out["issue_year"] = out["issue_d_dt"].dt.year.astype("Int16")
        out["issue_quarter"] = out["issue_d_dt"].dt.to_period("Q").astype("string")
        out["issue_month"] = out["issue_d_dt"].dt.to_period("M").astype("string")

    if {"issue_d_dt", "earliest_cr_line_dt"}.issubset(out.columns):
        out["credit_history_years"] = (out["issue_d_dt"] - out["earliest_cr_line_dt"]).dt.days / 365.25

    numeric_candidates = [
        "loan_amnt", "installment", "annual_inc", "dti", "delinq_2yrs", "inq_last_6mths",
        "mths_since_last_delinq", "mths_since_last_record", "open_acc", "pub_rec", "revol_bal",
        "total_acc", "collections_12_mths_ex_med", "mths_since_last_major_derog", "acc_now_delinq",
        "tot_coll_amt", "tot_cur_bal", "annual_inc_joint", "dti_joint", "total_rev_hi_lim",
        "acc_open_past_24mths", "avg_cur_bal", "bc_open_to_buy", "bc_util", "mort_acc",
        "pub_rec_bankruptcies", "tax_liens", "tot_hi_cred_lim", "total_bal_ex_mort", "total_bc_limit",
        "total_il_high_credit_limit", "out_prncp", "total_pymnt", "total_rec_prncp", "total_rec_int",
        "recoveries", "collection_recovery_fee", "last_pymnt_amnt", "last_fico_range_high", "last_fico_range_low",
    ]
    for col in numeric_candidates:
        if col in out:
            out[col] = pd.to_numeric(out[col], errors="coerce")

    categorical_cols = [
        "grade", "sub_grade", "emp_length", "home_ownership", "verification_status", "loan_status",
        "purpose", "zip_code", "addr_state", "initial_list_status", "application_type",
        "verification_status_joint", "disbursement_method", "hardship_flag", "debt_settlement_flag",
    ]
    for col in categorical_cols:
        if col in out:
            out[col] = out[col].astype("string").str.strip()

    return out


accepted = add_clean_features(accepted_raw)
print("Cleaned sample shape:", accepted.shape)
display(accepted.head())


In [ ]:
def suspicious_value_report(df: pd.DataFrame) -> pd.DataFrame:
    rules = {
        "loan_amnt": lambda s: (s <= 0) | (s > 100_000),
        "annual_inc": lambda s: (s < 0) | (s > 10_000_000),
        "dti": lambda s: (s < 0) | (s > 100),
        "int_rate_clean": lambda s: (s <= 0) | (s > 40),
        "revol_util_clean": lambda s: (s < 0) | (s > 200),
        "fico_mean": lambda s: (s < 300) | (s > 900),
        "credit_history_years": lambda s: (s < 0) | (s > 90),
        "open_acc": lambda s: s < 0,
        "pub_rec": lambda s: s < 0,
    }
    rows = []
    for col, rule in rules.items():
        if col in df:
            s = pd.to_numeric(df[col], errors="coerce")
            mask = rule(s) & s.notna()
            rows.append({
                "column": col,
                "suspicious_count": int(mask.sum()),
                "suspicious_pct": round(mask.mean() * 100, 4),
                "min": s.min(),
                "p01": s.quantile(0.01),
                "median": s.median(),
                "p99": s.quantile(0.99),
                "max": s.max(),
            })
    return pd.DataFrame(rows).sort_values("suspicious_pct", ascending=False)

quality_report = suspicious_value_report(accepted)
display_and_save_table(quality_report, "accepted_suspicious_value_report")


## 5.1 Structural Data Quality Checks

Before interpreting missingness or modeling readiness, check basic structural quality:

- **Column names**: identify leading/trailing spaces, uppercase letters, spaces, punctuation, or duplicate names. The accepted LendingClub file mostly uses lowercase snake_case already, so the EDA does not rename all accepted columns globally. If a future source file includes spaces or mixed casing, normalize with lowercase snake_case before feature engineering.
- **Text/categorical values**: strip extra whitespace from categorical values. The current cleaning function strips whitespace, but it does not globally collapse mixed-case categories because LendingClub categories are already mostly standardized. If mixed casing appears, add explicit category maps rather than blindly lowercasing every business label.
- **Data types**: convert percentages, terms, employment length, dates, FICO ranges, and numeric credit fields into analysis-ready types. Preserve raw fields and add cleaned helper fields where useful.

This section is a structural audit. It should flag issues for review; cleaning decisions remain explicit in `add_clean_features`.


In [ ]:
def structural_column_name_audit(columns: list[str]) -> pd.DataFrame:
    rows = []
    seen = {}
    for col in columns:
        seen[col] = seen.get(col, 0) + 1
        rows.append({
            "column": col,
            "has_leading_or_trailing_space": col != col.strip(),
            "has_uppercase": bool(re.search(r"[A-Z]", col)),
            "has_internal_space": " " in col,
            "has_non_snake_chars": bool(re.search(r"[^a-z0-9_]", col)),
            "snake_case_candidate": slugify(col),
            "duplicate_count": seen[col],
        })
    return pd.DataFrame(rows)


def categorical_value_audit(df: pd.DataFrame, columns: list[str], top_n: int = 12) -> pd.DataFrame:
    rows = []
    for col in columns:
        if col not in df:
            continue
        raw = df[col].astype("string")
        stripped = raw.str.strip()
        rows.append({
            "column": col,
            "n_unique_raw": int(raw.nunique(dropna=True)),
            "n_unique_after_strip": int(stripped.nunique(dropna=True)),
            "values_changed_by_strip": int((raw.notna() & stripped.notna() & raw.ne(stripped)).sum()),
            "has_case_collisions_after_lower": int(stripped.dropna().str.lower().nunique() < stripped.dropna().nunique()),
            "top_values": "; ".join([f"{idx}: {val}" for idx, val in stripped.value_counts(dropna=False).head(top_n).items()]),
        })
    return pd.DataFrame(rows)


def structural_type_audit(df: pd.DataFrame) -> pd.DataFrame:
    checks = [
        {"field": "int_rate", "expected_clean_field": "int_rate_clean", "expected_type": "numeric percentage"},
        {"field": "revol_util", "expected_clean_field": "revol_util_clean", "expected_type": "numeric percentage"},
        {"field": "term", "expected_clean_field": "term_months", "expected_type": "integer months"},
        {"field": "emp_length", "expected_clean_field": "emp_length_years", "expected_type": "numeric years"},
        {"field": "issue_d", "expected_clean_field": "issue_d_dt", "expected_type": "datetime"},
        {"field": "earliest_cr_line", "expected_clean_field": "earliest_cr_line_dt", "expected_type": "datetime"},
        {"field": "fico_range_low/fico_range_high", "expected_clean_field": "fico_mean", "expected_type": "numeric score"},
    ]
    rows = []
    for check in checks:
        clean = check["expected_clean_field"]
        rows.append({
            **check,
            "clean_field_present": clean in df.columns,
            "clean_field_dtype": str(df[clean].dtype) if clean in df.columns else "missing",
            "clean_field_missing_pct": round(df[clean].isna().mean() * 100, 2) if clean in df.columns else np.nan,
        })
    return pd.DataFrame(rows)


column_name_audit = structural_column_name_audit(accepted_columns)
display(column_name_audit[
    column_name_audit[[
        "has_leading_or_trailing_space",
        "has_uppercase",
        "has_internal_space",
        "has_non_snake_chars",
    ]].any(axis=1)
].head(40))

categorical_structural_audit = categorical_value_audit(
    accepted,
    [c for c in ["grade", "sub_grade", "home_ownership", "verification_status", "purpose", "addr_state", "loan_status", "application_type"] if c in accepted],
)
display(categorical_structural_audit)

type_structural_audit = structural_type_audit(accepted)
display(type_structural_audit)


## 5.2 Irrelevant Data for Repayment Prediction

Some columns should be dropped from a repayment-risk feature matrix because they do not describe borrower repayment capacity or credit risk at decision time. This is separate from the leakage audit: leakage fields are dangerous because they contain future information, while irrelevant fields are unhelpful identifiers, unavailable fields, metadata, or raw fields superseded by cleaned helper features.

Decision rule:

- Drop identifiers and lookup metadata such as `id`, `member_id`, and `url` from model features.
- Drop target/outcome helper columns from model features, including `loan_status`, `target_bad`, and `target_definition`.
- Drop fully unavailable or constant fields unless there is a reviewed business reason to keep them.
- Prefer cleaned helper fields over raw string versions for modeling, for example use `term_months` instead of `term`, `emp_length_years` instead of `emp_length`, `int_rate_clean` instead of raw percentage strings when applicable, and `issue_d_dt` only for split/vintage logic rather than as a direct model feature.
- Treat free-text fields such as `desc`, `title`, and `emp_title` as excluded from first-pass modeling unless a separate NLP, privacy, and fair-lending review is completed.

The table below is an audit table for the current EDA dataframe. It does not modify the raw source file.


In [ ]:
IRRELEVANT_IDENTIFIER_OR_METADATA = [
    "id", "member_id", "url",
]

IRRELEVANT_TARGET_OR_OUTCOME = [
    "loan_status", "target_bad", "target_definition",
]

RAW_FIELDS_SUPERSEDED_BY_CLEAN_HELPERS = {
    "term": "term_months",
    "emp_length": "emp_length_years",
    "int_rate": "int_rate_clean",
    "revol_util": "revol_util_clean",
    "issue_d": "issue_d_dt / issue_year / issue_month",
    "earliest_cr_line": "earliest_cr_line_dt / credit_history_years",
    "fico_range_low": "fico_mean",
    "fico_range_high": "fico_mean",
}

TEXT_FIELDS_REVIEW_BEFORE_USE = [
    "desc", "title", "emp_title",
]


def irrelevant_feature_audit(df: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for col in df.columns:
        reason = None
        recommendation = "candidate_or_review_elsewhere"
        replacement = ""

        if col in IRRELEVANT_IDENTIFIER_OR_METADATA:
            reason = "identifier_or_lookup_metadata"
            recommendation = "drop_from_model_features"
        elif col in IRRELEVANT_TARGET_OR_OUTCOME:
            reason = "target_or_outcome_column"
            recommendation = "drop_from_model_features"
        elif col in RAW_FIELDS_SUPERSEDED_BY_CLEAN_HELPERS:
            reason = "raw_field_superseded_by_clean_helper"
            recommendation = "prefer_clean_helper_for_modeling"
            replacement = RAW_FIELDS_SUPERSEDED_BY_CLEAN_HELPERS[col]
        elif col in TEXT_FIELDS_REVIEW_BEFORE_USE:
            reason = "free_text_requires_privacy_nlp_fair_lending_review"
            recommendation = "exclude_from_first_pass_model"
        elif df[col].isna().all():
            reason = "fully_missing_in_working_dataframe"
            recommendation = "drop_from_model_features"
        elif df[col].nunique(dropna=True) <= 1:
            reason = "constant_or_near_constant_in_working_dataframe"
            recommendation = "drop_from_model_features_unless_business_review_keeps"

        if reason:
            rows.append({
                "column": col,
                "reason": reason,
                "recommendation": recommendation,
                "replacement_or_allowed_use": replacement,
                "missing_pct": round(df[col].isna().mean() * 100, 2),
                "n_unique_non_null": int(df[col].nunique(dropna=True)),
                "dtype": str(df[col].dtype),
            })

    return pd.DataFrame(rows).sort_values(["recommendation", "reason", "column"])


irrelevant_data_audit = irrelevant_feature_audit(accepted)
display(irrelevant_data_audit)


## 5.3 Missing Data Label Standardization

Some datasets encode missing values as text labels rather than true nulls. Examples include empty strings, `N/A`, `NA`, `NO DATA`, `NULL`, `nan`, `MISSING`, and `UNKNOWN`. These labels should be standardized to `pd.NA` / `np.nan` so pandas missingness checks, plots, and modeling-prep logic recognize them.

Decision rule:

- Apply this only to text-like columns in the working EDA dataframe.
- Strip leading/trailing whitespace before checking missing-label tokens.
- Use a conservative token list. Do not automatically convert business categories such as `NONE`, because `NONE` can be a legitimate LendingClub category in fields like home ownership.
- Do not modify the raw LendingClub source file.


In [ ]:
STANDARD_MISSING_LABELS = {
    "",
    "N/A",
    "NA",
    "NO DATA",
    "NULL",
    "NAN",
    "MISSING",
    "UNKNOWN",
}


def standardize_text_missing_labels(df: pd.DataFrame, missing_labels: set[str] = STANDARD_MISSING_LABELS) -> tuple[pd.DataFrame, pd.DataFrame]:
    out = df.copy()
    rows = []
    text_cols = out.select_dtypes(include=["object", "string"]).columns

    for col in text_cols:
        original_missing = int(out[col].isna().sum())
        stripped = out[col].astype("string").str.strip()
        token_mask = stripped.notna() & stripped.str.upper().isin(missing_labels)
        converted_count = int(token_mask.sum())

        if converted_count:
            out[col] = stripped.mask(token_mask, pd.NA)
        else:
            out[col] = stripped

        rows.append({
            "column": col,
            "original_missing_count": original_missing,
            "standardized_missing_label_count": converted_count,
            "missing_count_after_standardization": int(out[col].isna().sum()),
            "n_unique_after_standardization": int(out[col].nunique(dropna=True)),
        })

    return out, pd.DataFrame(rows).sort_values("standardized_missing_label_count", ascending=False)


accepted, missing_label_standardization_summary = standardize_text_missing_labels(accepted)
display(missing_label_standardization_summary)


## 5.4 Duplicate Row Removal

Exact duplicate records can overweight repeated loans and distort missingness, portfolio counts, bad-rate summaries, and model training. Remove exact duplicate rows from the working EDA dataframe before target construction and downstream analysis.

Decision rule:

- Use `accepted.duplicated()` to count exact duplicate rows across all columns currently in the working dataframe.
- Use `accepted.drop_duplicates()` to remove exact duplicate records.
- This does not modify the raw LendingClub source file.
- This only removes fully identical records. It does not collapse records that share an `id` but differ in any field; those require separate business/data-lineage review.


In [ ]:
duplicate_rows_before = len(accepted)
exact_duplicate_count = int(accepted.duplicated().sum())

duplicate_removal_summary = pd.DataFrame([
    {
        "scope": "accepted_working_dataframe",
        "duplicate_rule": "drop exact duplicate rows across all current dataframe columns",
        "rows_before": duplicate_rows_before,
        "exact_duplicate_rows": exact_duplicate_count,
        "rows_after": duplicate_rows_before - exact_duplicate_count,
        "pct_removed": round(exact_duplicate_count / duplicate_rows_before * 100, 4)
        if duplicate_rows_before
        else np.nan,
    }
])
display(duplicate_removal_summary)

accepted = accepted.drop_duplicates().reset_index(drop=True)
print("Accepted working dataframe rows after exact duplicate removal:", len(accepted))


## 5.5 Imputation Plan

Imputation should be planned during EDA but applied later in the modeling-prep pipeline. Applying imputation inside EDA would hide real missingness patterns and make the missingness analysis harder to audit.

Decision rule for the later modeling-prep notebook:

- For numerical fields, use median imputation by default because credit-risk variables such as income, balances, DTI, and utilization are often skewed.
- Mean imputation can be considered only for approximately symmetric numerical fields after distribution review.
- For categorical fields, use mode imputation by default.
- For fields where missingness may be informative, add a missingness indicator before imputation.
- For structural missingness, such as joint-applicant fields missing for individual applications, handle conditionally by `application_type` rather than globally imputing as ordinary missingness.

The table below prepares candidate imputation values for review. It does not modify `accepted`.


In [ ]:
NUMERIC_IMPUTATION_CANDIDATES = [
    c for c in [
        "annual_inc", "dti", "fico_mean", "revol_util_clean", "emp_length_years",
        "mort_acc", "bc_util", "mths_since_last_delinq", "mths_since_last_record",
    ]
    if c in accepted
]

CATEGORICAL_IMPUTATION_CANDIDATES = [
    c for c in [
        "home_ownership", "verification_status", "purpose", "application_type",
        "grade", "sub_grade",
    ]
    if c in accepted
]

MISSINGNESS_INDICATOR_RECOMMENDED = [
    c for c in [
        "emp_length_years", "bc_util", "mths_since_last_delinq",
        "mths_since_last_record", "annual_inc_joint", "dti_joint",
    ]
    if c in accepted
]

STRUCTURAL_MISSINGNESS_FIELDS = [
    c for c in ["annual_inc_joint", "dti_joint", "verification_status_joint"] if c in accepted
]


def build_imputation_plan(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for col in NUMERIC_IMPUTATION_CANDIDATES:
        s = pd.to_numeric(df[col], errors="coerce")
        rows.append({
            "column": col,
            "feature_type": "numeric",
            "missing_count": int(s.isna().sum()),
            "missing_pct": round(s.isna().mean() * 100, 2),
            "recommended_imputation": "median",
            "median_value": s.median(),
            "mean_value": s.mean(),
            "mode_value": np.nan,
            "missing_indicator_recommended": col in MISSINGNESS_INDICATOR_RECOMMENDED,
            "structural_missingness_note": "",
        })

    for col in CATEGORICAL_IMPUTATION_CANDIDATES:
        s = df[col].astype("string")
        mode_values = s.mode(dropna=True)
        rows.append({
            "column": col,
            "feature_type": "categorical",
            "missing_count": int(s.isna().sum()),
            "missing_pct": round(s.isna().mean() * 100, 2),
            "recommended_imputation": "mode",
            "median_value": np.nan,
            "mean_value": np.nan,
            "mode_value": mode_values.iloc[0] if len(mode_values) else pd.NA,
            "missing_indicator_recommended": col in MISSINGNESS_INDICATOR_RECOMMENDED,
            "structural_missingness_note": "",
        })

    for col in STRUCTURAL_MISSINGNESS_FIELDS:
        s = df[col]
        rows.append({
            "column": col,
            "feature_type": "structural",
            "missing_count": int(s.isna().sum()),
            "missing_pct": round(s.isna().mean() * 100, 2),
            "recommended_imputation": "conditional_by_application_type_or_exclude",
            "median_value": np.nan,
            "mean_value": np.nan,
            "mode_value": pd.NA,
            "missing_indicator_recommended": True,
            "structural_missingness_note": "Joint-applicant field; missing for Individual applications is expected.",
        })

    return pd.DataFrame(rows).sort_values(["feature_type", "missing_pct", "column"], ascending=[True, False, True])


imputation_plan = build_imputation_plan(accepted)
display(imputation_plan)


## 5.6 High-Missingness Feature Decision Framework

High missingness alone is not enough reason to drop a field. In credit-risk modeling, the correct question is whether the field helps predict repayment after leakage, target, identifier, and compliance exclusions have already been applied.

This EDA section defines the decision framework. The actual feature-importance test should be run later in `Accepted_Loan_Modeling_Prep.ipynb` or the modeling notebook, using a leakage-clean baseline model.

### Step 1: Feature-Importance Save Test

Before deciding how to impute or transform a high-missingness column, test whether the column matters for the repayment target.

- Train a quick baseline tree-based model that can handle missing values natively or robustly, such as LightGBM, HistGradientBoosting, or a Random Forest preprocessing pipeline.
- Use only leakage-clean, application-time candidate features.
- Rank features by gain/importances/permutation importance on a validation split.
- If a high-missingness feature has zero or negligible importance, drop it from the modeling feature set.
- If the feature ranks meaningfully for predicting `target_bad`, move it to Step 2.

### Step 2: Inversion Strategy for Saved High-Missingness Features

For columns with massive missingness, ordinary mean or median imputation can destroy the feature's distribution. If the model says the feature matters, treat missingness as the primary signal instead of hiding it.

Action A: categorical pivot.

- Convert the feature into a categorical representation.
- Put missing values into a dominant category such as `Not_Reported`.
- Bin observed numeric values into interpretable groups such as `Low`, `Medium`, and `High`.

Action B: strict indicator method.

- Create a binary missingness indicator: `1` when the original value is missing, `0` otherwise.
- Fill missing numeric values with a distinct out-of-distribution placeholder such as `-999`, or fill missing text values with `Unknown`.
- Use this only with models and explainability workflows that can safely handle sentinel values.

### Step 3: Domain-Knowledge Feature

When missingness has business meaning, turn the gap into an explicit feature. Examples:

- Joint-applicant fields missing because the application is individual.
- Delinquency-recency fields missing because no delinquency is recorded or because bureau history is unavailable.
- Employment length missing because the borrower did not report employment history.

Decision: high-missingness fields are not automatically dropped and not blindly median/mode imputed. They must pass the feature-importance save test first. If saved, use inversion, indicators, or domain-specific categorical features in modeling prep.


In [ ]:
HIGH_MISSINGNESS_REVIEW_THRESHOLD = 50.0

high_missingness_feature_framework = pd.DataFrame([
    {
        "step": "1_feature_importance_save_test",
        "question": "Does the high-missingness feature actually help predict target_bad?",
        "recommended_test": "Baseline leakage-clean tree model plus validation feature importance.",
        "drop_condition": "Drop if feature importance is zero or negligible.",
        "save_condition": "Keep for transformation review if feature importance is meaningful and stable.",
    },
    {
        "step": "2a_categorical_pivot",
        "question": "Can missingness become an explicit category?",
        "recommended_test": "Create Not_Reported plus Low/Medium/High bins for observed values.",
        "drop_condition": "Drop if binned feature has no predictive or business value.",
        "save_condition": "Use if missing/not-reported status is predictive and explainable.",
    },
    {
        "step": "2b_strict_indicator_method",
        "question": "Should missingness and observed magnitude be separated?",
        "recommended_test": "Create missing flag and fill original with a distinct sentinel value.",
        "drop_condition": "Drop if sentinel approach harms validation stability or explanations.",
        "save_condition": "Use if model performance and reasonability improve under validation.",
    },
    {
        "step": "3_domain_knowledge_feature",
        "question": "Does the missingness pattern have known business meaning?",
        "recommended_test": "Map missingness to an explicit domain category or rule.",
        "drop_condition": "Drop if the domain meaning is unsupported or compliance-risky.",
        "save_condition": "Use if the feature is defensible, stable, and explainable.",
    },
])
display(high_missingness_feature_framework)

high_missingness_review = pd.DataFrame({
    "column": accepted.columns,
    "missing_count": accepted.isna().sum().values,
    "missing_pct": accepted.isna().mean().mul(100).round(2).values,
    "dtype": accepted.dtypes.astype(str).values,
})
high_missingness_review = high_missingness_review[
    high_missingness_review["missing_pct"] > HIGH_MISSINGNESS_REVIEW_THRESHOLD
].sort_values(["missing_pct", "column"], ascending=[False, True])
high_missingness_review["eda_decision"] = (
    "do_not_auto_drop; send to feature-importance save test in modeling prep"
)
display(high_missingness_review)


## 6. Target Definition for Accepted Loans

The target is defined only from accepted-loan performance outcomes. `Fully Paid` is good. Charged-off/default/late hardship-like terminal or delinquent statuses are bad. `Current`, `Issued`, and grace-period records are censored/unresolved for completed-loan modeling.

For repayment-risk modeling, train on completed loans only unless a survival/hazard framework is explicitly chosen.


In [ ]:
GOOD_STATUSES = {"Fully Paid", "Does not meet the credit policy. Status:Fully Paid"}
BAD_STATUSES = {
    "Charged Off",
    "Default",
    "Does not meet the credit policy. Status:Charged Off",
    "Late (31-120 days)",
    "Late (16-30 days)",
}
CENSORED_STATUSES = {"Current", "In Grace Period", "Issued"}


def add_target(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    status = out["loan_status"].astype("string").str.strip()
    out["target_bad"] = np.select(
        [status.isin(BAD_STATUSES), status.isin(GOOD_STATUSES)],
        [1.0, 0.0],
        default=np.nan,
    )
    out["target_definition"] = np.select(
        [status.isin(BAD_STATUSES), status.isin(GOOD_STATUSES), status.isin(CENSORED_STATUSES)],
        ["bad_terminal_or_delinquent", "good_terminal", "censored_unresolved"],
        default="review_required",
    )
    return out


accepted = add_target(accepted)
completed = accepted[accepted["target_bad"].notna()].copy()

status_summary = (
    accepted.groupby(["loan_status", "target_definition"], dropna=False)
    .size()
    .reset_index(name="n")
    .sort_values("n", ascending=False)
)
status_summary["pct_sample"] = (status_summary["n"] / len(accepted) * 100).round(2)
display_and_save_table(status_summary, "accepted_loan_status_target_mapping")

print("Completed-modeling sample rows:", len(completed))
print("Observed completed-loan bad rate:", round(completed["target_bad"].mean(), 4) if len(completed) else np.nan)


## 6.1 Full-Dataset Loan-Status Imbalance

The target mapping above is based on the memory-aware working sample. For class-imbalance reporting, scan the full accepted-loan file and count every `loan_status` value. This avoids confusing `pct_sample` with missingness and gives the project a defensible full-population imbalance table.


In [ ]:
def repayment_risk_bucket_from_status(status: str) -> str:
    if status in GOOD_STATUSES:
        return "low_observed_repayment_risk"
    if status in {"Late (16-30 days)", "In Grace Period"}:
        return "medium_current_or_early_delinquency_review"
    if status in {
        "Charged Off",
        "Default",
        "Does not meet the credit policy. Status:Charged Off",
        "Late (31-120 days)",
    }:
        return "high_observed_repayment_risk"
    if status in {"Current", "Issued"}:
        return "unresolved_censored_not_ranked"
    if pd.isna(status):
        return "missing_status"
    return "review_required"


def target_definition_from_status(status: str) -> str:
    if status in GOOD_STATUSES:
        return "good_terminal"
    if status in BAD_STATUSES:
        return "bad_terminal_or_delinquent"
    if status in CENSORED_STATUSES:
        return "censored_unresolved"
    if pd.isna(status):
        return "missing_status"
    return "review_required"


full_status_counts = None
full_status_rows = 0
full_status_missing = 0

for chunk in pd.read_csv(ACCEPTED_PATH, usecols=["loan_status"], chunksize=CONFIG.chunk_size, low_memory=False):
    status = chunk["loan_status"].astype("string").str.strip()
    full_status_rows += len(status)
    full_status_missing += int(status.isna().sum())
    counts = status.value_counts(dropna=False)
    full_status_counts = counts if full_status_counts is None else full_status_counts.add(counts, fill_value=0)

full_status_distribution = full_status_counts.rename_axis("loan_status").reset_index(name="n")
full_status_distribution["n"] = full_status_distribution["n"].astype(int)
full_status_distribution["target_definition"] = full_status_distribution["loan_status"].apply(target_definition_from_status)
full_status_distribution["repayment_risk_bucket"] = full_status_distribution["loan_status"].apply(repayment_risk_bucket_from_status)
full_status_distribution["pct_full_dataset"] = (full_status_distribution["n"] / full_status_rows * 100).round(4)
full_status_distribution = full_status_distribution[
    ["loan_status", "target_definition", "repayment_risk_bucket", "n", "pct_full_dataset"]
].sort_values("n", ascending=False)

display_and_save_table(full_status_distribution, "accepted_loan_status_full_distribution")

full_status_risk_bucket = (
    full_status_distribution
    .groupby(["target_definition", "repayment_risk_bucket"], as_index=False)["n"]
    .sum()
)
full_status_risk_bucket["pct_full_dataset"] = (full_status_risk_bucket["n"] / full_status_rows * 100).round(4)
full_status_risk_bucket = full_status_risk_bucket.sort_values("n", ascending=False)
display_and_save_table(full_status_risk_bucket, "accepted_loan_status_full_distribution_by_risk_bucket")

print("Full accepted rows scanned:", full_status_rows)
print("Missing loan_status rows:", full_status_missing)
print("Missing loan_status pct:", round(full_status_missing / full_status_rows * 100, 4))


## 6.2 Remove Current Loans From Modeling EDA

`Current` loans are active loans without final repayment outcomes. They should not be treated as low-risk/good outcomes. For the modeling-oriented EDA sections below, remove rows where `loan_status == "Current"` from the working dataframe. The raw source file remains unchanged; this is an analysis-frame filter.


In [ ]:
CURRENT_STATUS = "Current"

current_mask = accepted["loan_status"].astype("string").str.strip().eq(CURRENT_STATUS)
working_rows_before_current_filter = len(accepted)
working_current_rows = int(current_mask.sum())
working_rows_after_current_filter = int((~current_mask).sum())

current_filter_summary_rows = [
    {
        "scope": "working_sample",
        "filter_rule": 'loan_status != "Current"',
        "rows_before": working_rows_before_current_filter,
        "current_rows_removed": working_current_rows,
        "rows_after": working_rows_after_current_filter,
        "pct_removed": round(working_current_rows / working_rows_before_current_filter * 100, 4)
        if working_rows_before_current_filter
        else np.nan,
    }
]

if "full_status_distribution" in globals() and "full_status_rows" in globals():
    full_current_rows = int(
        full_status_distribution.loc[
            full_status_distribution["loan_status"].astype("string").eq(CURRENT_STATUS),
            "n",
        ].sum()
    )
    current_filter_summary_rows.append(
        {
            "scope": "full_dataset_reference",
            "filter_rule": 'loan_status != "Current"',
            "rows_before": int(full_status_rows),
            "current_rows_removed": full_current_rows,
            "rows_after": int(full_status_rows - full_current_rows),
            "pct_removed": round(full_current_rows / full_status_rows * 100, 4)
            if full_status_rows
            else np.nan,
        }
    )

current_filter_summary = pd.DataFrame(current_filter_summary_rows)
display_and_save_table(current_filter_summary, "accepted_current_status_removal_summary")

accepted = accepted.loc[~current_mask].copy()
completed = accepted[accepted["target_bad"].notna()].copy()

print("Removed Current rows from accepted working dataframe:", working_current_rows)
print("Accepted working dataframe rows after Current removal:", len(accepted))
print("Completed modeling rows after Current removal:", len(completed))


## 7. Missingness Analysis

Missingness is evaluated on the post-`Current`-removal working sample. The histogram below is the main missingness overview: it ranks dataframe fields by the percentage of rows that are null.

The first table reports one row per dataframe column. `column` is the field name in the EDA dataframe, `missing_count` is the number of null values, `missing_pct` is the percentage of rows that are null, `dtype` is the pandas data type after EDA cleaning, and `non_null` is the number of populated rows. These percentages describe data availability; they do not directly mean credit risk or target class imbalance.

The slice tables after the histogram answer a different question: whether missingness is concentrated in specific years, grades, application types, or loan statuses. They use a wide format for readability. Each row is a slice value, `n_rows` is the denominator for that slice, every feature column ending in `_missing_count` is the number of null rows, and every feature column ending in `_missing_pct` is the percent of rows in that slice where the feature is null.

Interpret high-missingness fields using the LendingClub data dictionary. For example, `member_id` is defined as a unique LendingClub-assigned borrower-member identifier. In this public accepted-loan file it is 100% missing, so it is not useful as a borrower-risk feature and should be excluded as an unavailable identifier. By contrast, fields such as `annual_inc_joint`, `dti_joint`, and `verification_status_joint` are joint-application fields, so high missingness is expected for individual applications. Fields such as `mths_since_last_record`, `mths_since_last_delinq`, and `mths_since_recent_bc_dlq` describe months since a prior public record or delinquency; missingness may indicate no such event, unavailable bureau history, or schema coverage differences rather than a simple data-quality failure.

Groups with small denominators are flagged in the slice tables. For example, `Default` has only 2 rows in the current working sample, so a 50% missing rate for `bc_util` means 1 missing value out of 2 loans. That is not enough evidence to infer a systematic missingness pattern.\n\nFor small groups, inspect counts before percentages. For example, a `dti_missing_pct` of 0.29% for `Late (16-30 days)` means 1 missing `dti` value out of 343 rows, not a strong standalone signal of systematic missingness.\n

### Section 7 Missingness Insights

The slice tables are diagnostic, not feature-selection rules. They show that most core fields have stable, low missingness across `loan_status`, but a few patterns deserve modeling attention:

- `emp_length_years` missingness is higher for `Charged Off` loans than for `Fully Paid` loans: 3,004 of 38,709 charged-off loans are missing employment length (7.76%) versus 8,503 of 148,646 fully paid loans (5.72%). This suggests employment-length missingness may carry weak risk signal and should be tested with a missingness indicator.
- `dti`, `revol_util_clean`, and `bc_util` have slightly higher missingness for `Charged Off` than `Fully Paid`, but the differences are very small. Treat them as data-quality checks, not strong standalone risk signals.
- `annual_inc_joint` and `dti_joint` are missing for almost all individual applications and populated for joint applications. This is structural missingness, so these fields should be handled conditionally on `application_type` or excluded from a first-pass model.
- Small groups should not drive modeling decisions. For example, `Default` has only 2 rows in the working sample, so its 50% `bc_util` missingness is 1 missing value out of 2 loans.

Modeling suggestion: keep missingness indicators for selected fields where missingness may be informative, especially `emp_length_years`, `bc_util`, and joint-application fields if joint applications are modeled. Do not blindly impute all missing values without preserving whether the value was missing.



In [ ]:
missingness = pd.DataFrame({
    "column": accepted.columns,
    "missing_count": accepted.isna().sum().values,
    "missing_pct": accepted.isna().mean().mul(100).round(2).values,
    "dtype": accepted.dtypes.astype(str).values,
    "non_null": accepted.notna().sum().values,
}).sort_values("missing_pct", ascending=False)

display_and_save_table(missingness, "accepted_working_sample_missingness")

plt.figure(figsize=(10, 8))
plot_df = missingness.head(35).sort_values("missing_pct")
sns.barplot(data=plot_df, x="missing_pct", y="column", color="#4C78A8")
plt.title("Accepted Loans: Highest Missingness in Working Sample")
plt.xlabel("Missing %")
plt.ylabel("")
save_current_plot("accepted_highest_missingness")
plt.show()


In [ ]:
important_slices = [c for c in ["issue_year", "grade", "application_type", "loan_status"] if c in accepted]
important_missing_cols = [
    c for c in [
        "annual_inc", "dti", "fico_mean", "revol_util_clean", "emp_length_years",
        "mort_acc", "bc_util", "annual_inc_joint", "dti_joint",
    ]
    if c in accepted
]


def missingness_pct_by_slice(
    df: pd.DataFrame,
    slice_col: str,
    feature_cols: list[str],
    min_stable_n: int = CONFIG.min_group_n,
) -> pd.DataFrame:
    """Return readable slice-level null counts and percentages with explicit denominators."""
    grouped = df.groupby(slice_col, dropna=False)
    counts = grouped.size().reset_index(name="n_rows")
    missing_counts = grouped[feature_cols].apply(lambda x: x.isna().sum()).reset_index()
    missing_pct = grouped[feature_cols].apply(lambda x: x.isna().mean().mul(100).round(2)).reset_index()

    missing_counts = missing_counts.rename(columns={feature: f"{feature}_missing_count" for feature in feature_cols})
    missing_pct = missing_pct.rename(columns={feature: f"{feature}_missing_pct" for feature in feature_cols})

    out = counts.merge(missing_counts, on=slice_col, how="left").merge(missing_pct, on=slice_col, how="left")
    out["small_group_flag"] = np.where(out["n_rows"] < min_stable_n, "unstable_small_n", "ok")

    ordered_cols = [slice_col, "n_rows", "small_group_flag"]
    for feature in feature_cols:
        ordered_cols.extend([f"{feature}_missing_count", f"{feature}_missing_pct"])
    return out[ordered_cols]


for slice_col in important_slices:
    slice_missing = missingness_pct_by_slice(accepted, slice_col, important_missing_cols)
    display_and_save_table(slice_missing, f"missingness_by_{slice_col}")


## 7.1 Automated Missingness Mechanism Screening

MCAR, MAR, and MNAR are missing-data mechanisms, but they cannot be proven fully from the observed table alone. In particular, MNAR means missingness depends on the unobserved value itself, so it requires domain evidence, collection-process evidence, or sensitivity analysis.

This section automates a screening step:

- **Consistent with MCAR**: missingness is low and does not vary materially across observed slices.
- **Potential MAR / structural missingness**: missingness varies by observed fields such as `application_type`, `loan_status`, `issue_year`, or `grade`.
- **MNAR not identifiable from observed data**: no automatic claim is made; use domain review when the missing field could be absent because of its unobserved value.

Decision rule: use this table to prioritize modeling treatment. Structural or MAR-like fields should preserve missingness indicators or conditional feature logic later in modeling prep. Fields that are merely consistent with MCAR can use simpler imputation, but this is still a modeling choice to validate.


In [ ]:
mechanism_candidate_cols = [
    c for c in [
        "annual_inc", "dti", "fico_mean", "revol_util_clean", "emp_length_years",
        "mort_acc", "bc_util", "annual_inc_joint", "dti_joint",
        "mths_since_last_record", "mths_since_last_delinq", "mths_since_recent_bc_dlq",
    ]
    if c in accepted
]

mechanism_slice_cols = [c for c in ["application_type", "loan_status", "issue_year", "grade"] if c in accepted]


def missing_pct_gap(df: pd.DataFrame, feature: str, slice_col: str, min_n: int = CONFIG.min_group_n) -> tuple[float, str]:
    grouped = (
        df.groupby(slice_col, dropna=False)[feature]
        .agg(group_rows="size", missing_pct=lambda x: x.isna().mean() * 100)
        .reset_index()
    )
    stable = grouped[grouped["group_rows"] >= min_n]
    if stable.empty or stable["missing_pct"].isna().all():
        return np.nan, "no_stable_groups"
    max_row = stable.loc[stable["missing_pct"].idxmax()]
    min_row = stable.loc[stable["missing_pct"].idxmin()]
    gap = round(float(max_row["missing_pct"] - min_row["missing_pct"]), 2)
    detail = f"{slice_col}: {max_row[slice_col]} {max_row['missing_pct']:.2f}% vs {min_row[slice_col]} {min_row['missing_pct']:.2f}%"
    return gap, detail


def target_missingness_gap(df: pd.DataFrame, feature: str) -> tuple[float, str]:
    if "target_bad" not in df:
        return np.nan, "target_bad unavailable"
    tmp = df[df["target_bad"].notna()].copy()
    if tmp.empty or tmp[feature].isna().sum() == 0:
        return np.nan, "no missing completed rows"
    missing = tmp[feature].isna()
    if missing.nunique() < 2:
        return np.nan, "missing indicator has one level"
    bad_when_missing = tmp.loc[missing, "target_bad"].mean()
    bad_when_observed = tmp.loc[~missing, "target_bad"].mean()
    gap = round(float((bad_when_missing - bad_when_observed) * 100), 2)
    detail = f"bad rate missing {bad_when_missing:.2%} vs observed {bad_when_observed:.2%}"
    return gap, detail


mechanism_rows = []
for feature in mechanism_candidate_cols:
    total_rows = len(accepted)
    missing_count = int(accepted[feature].isna().sum())
    missing_pct = round(missing_count / total_rows * 100, 2) if total_rows else np.nan

    slice_gaps = {}
    slice_details = {}
    for slice_col in mechanism_slice_cols:
        gap, detail = missing_pct_gap(accepted, feature, slice_col)
        slice_gaps[slice_col] = gap
        slice_details[slice_col] = detail

    target_gap, target_detail = target_missingness_gap(accepted, feature)
    valid_gaps = [v for v in slice_gaps.values() if not pd.isna(v)]
    max_slice_gap = round(max(valid_gaps), 2) if valid_gaps else np.nan

    structural_reason = ""
    if feature.endswith("_joint") and "application_type" in slice_gaps and slice_gaps["application_type"] >= 80:
        mechanism_screen = "structural_missingness"
        structural_reason = "Joint-applicant field; missingness is expected for Individual applications."
        recommended_action = "Handle conditionally by application_type or exclude from first-pass model."
    elif (not pd.isna(max_slice_gap) and max_slice_gap >= 5) or (not pd.isna(target_gap) and abs(target_gap) >= 2):
        mechanism_screen = "potential_MAR_observed_pattern"
        recommended_action = "Preserve missingness indicator later and validate imputation/model impact."
    elif missing_count == 0:
        mechanism_screen = "no_missingness_observed"
        recommended_action = "No missingness action needed in this working sample."
    else:
        mechanism_screen = "consistent_with_MCAR_screen"
        recommended_action = "Simple imputation may be acceptable; validate during modeling."

    mechanism_rows.append({
        "feature": feature,
        "missing_count": missing_count,
        "missing_pct": missing_pct,
        "max_observed_slice_gap_pct_points": max_slice_gap,
        "application_type_gap_detail": slice_details.get("application_type", ""),
        "loan_status_gap_detail": slice_details.get("loan_status", ""),
        "issue_year_gap_detail": slice_details.get("issue_year", ""),
        "grade_gap_detail": slice_details.get("grade", ""),
        "target_bad_rate_gap_pct_points": target_gap,
        "target_gap_detail": target_detail,
        "mechanism_screen": mechanism_screen,
        "structural_reason": structural_reason,
        "mnar_note": "MNAR not identifiable from observed data alone.",
        "recommended_action": recommended_action,
    })

missingness_mechanism_screening = pd.DataFrame(mechanism_rows).sort_values(
    ["mechanism_screen", "missing_pct"], ascending=[True, False]
)
display_and_save_table(missingness_mechanism_screening, "accepted_missingness_mechanism_screening")


## 8. Univariate Portfolio EDA

This section describes accepted-loan origination amounts, pricing, borrower credit profile, terms, purposes, and grade mix.


In [ ]:
NUMERIC_EDA_COLS = [
    "loan_amnt", "int_rate_clean", "installment", "annual_inc", "dti", "fico_mean",
    "emp_length_years", "credit_history_years", "open_acc", "revol_bal", "revol_util_clean",
    "total_acc", "mort_acc", "bc_util", "total_rev_hi_lim", "tot_cur_bal",
]
NUMERIC_EDA_COLS = [c for c in NUMERIC_EDA_COLS if c in accepted]

numeric_summary = accepted[NUMERIC_EDA_COLS].describe(percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]).T
numeric_summary["missing_pct"] = accepted[NUMERIC_EDA_COLS].isna().mean().mul(100).round(2)
display_and_save_table(numeric_summary.reset_index(names="column"), "accepted_numeric_summary")


In [ ]:
CATEGORICAL_EDA_COLS = [
    "term", "grade", "sub_grade", "home_ownership", "verification_status", "purpose",
    "addr_state", "application_type", "initial_list_status", "disbursement_method",
]
CATEGORICAL_EDA_COLS = [c for c in CATEGORICAL_EDA_COLS if c in accepted]

for col in CATEGORICAL_EDA_COLS:
    print(f"\n{col}")
    display_and_save_table(categorical_summary(accepted, col, top_n=25).reset_index(names=col), f"categorical_summary_{col}")


In [ ]:
plot_numeric = [c for c in ["loan_amnt", "int_rate_clean", "annual_inc", "dti", "fico_mean", "revol_util_clean"] if c in accepted]
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()
for ax, col in zip(axes, plot_numeric):
    s = accepted[col].dropna()
    if s.empty:
        ax.set_visible(False)
        continue
    lo, hi = s.quantile([0.01, 0.99])
    sns.histplot(s.clip(lo, hi), bins=40, ax=ax, color="#4C78A8")
    ax.set_title(f"{col} (p01-p99 clipped)")
for ax in axes[len(plot_numeric):]:
    ax.set_visible(False)
save_current_plot("accepted_numeric_distributions")
plt.show()


In [ ]:
for col in ["grade", "purpose", "home_ownership", "verification_status", "term"]:
    if col not in accepted:
        continue
    plt.figure(figsize=(10, 5))
    order = accepted[col].value_counts().head(20).index
    sns.countplot(data=accepted, y=col, order=order, color="#4C78A8")
    plt.title(f"Accepted Loans: {col} Distribution")
    plt.xlabel("Count in sample")
    plt.ylabel("")
    save_current_plot(f"accepted_{col}_distribution")
    plt.show()


## 9. Temporal and Vintage EDA

Credit-risk data is not IID over time. LendingClub mix, pricing, macro conditions, and underwriting strategy changed materially from 2007 to 2018. Model validation should therefore use time-based splits.


In [ ]:
if "issue_year" in accepted:
    yearly = (
        accepted.groupby("issue_year", dropna=True)
        .agg(
            sample_loans=("id", "count") if "id" in accepted else ("loan_amnt", "count"),
            avg_loan_amnt=("loan_amnt", "mean"),
            avg_int_rate=("int_rate_clean", "mean"),
            avg_fico=("fico_mean", "mean"),
            avg_dti=("dti", "mean"),
        )
        .reset_index()
    )
    yearly[["avg_loan_amnt", "avg_int_rate", "avg_fico", "avg_dti"]] = yearly[["avg_loan_amnt", "avg_int_rate", "avg_fico", "avg_dti"]].round(2)
    display_and_save_table(yearly, "accepted_yearly_portfolio_summary")

    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
    sns.lineplot(data=yearly, x="issue_year", y="sample_loans", marker="o", ax=axes[0, 0])
    axes[0, 0].set_title("Origination Volume in Sample")
    sns.lineplot(data=yearly, x="issue_year", y="avg_loan_amnt", marker="o", ax=axes[0, 1])
    axes[0, 1].set_title("Average Loan Amount")
    sns.lineplot(data=yearly, x="issue_year", y="avg_int_rate", marker="o", ax=axes[1, 0])
    axes[1, 0].set_title("Average Interest Rate")
    sns.lineplot(data=yearly, x="issue_year", y="avg_fico", marker="o", ax=axes[1, 1])
    axes[1, 1].set_title("Average FICO Mean")
    save_current_plot("accepted_temporal_portfolio_summary")
    plt.show()


In [ ]:
if {"issue_year", "grade"}.issubset(accepted.columns):
    grade_time = (
        accepted.groupby(["issue_year", "grade"], observed=True)
        .size()
        .reset_index(name="n")
    )
    grade_time["pct_within_year"] = grade_time.groupby("issue_year")["n"].transform(lambda x: x / x.sum() * 100)
    display_and_save_table(grade_time, "accepted_grade_mix_by_year")

    plt.figure(figsize=(12, 6))
    sns.lineplot(data=grade_time, x="issue_year", y="pct_within_year", hue="grade", marker="o")
    plt.title("Accepted Loans: Grade Mix Over Time")
    plt.ylabel("% within issue year")
    plt.xlabel("Issue year")
    save_current_plot("accepted_grade_mix_over_time")
    plt.show()


## 10. Outcome EDA and Observed Bad Rates

Observed bad-rate analysis is restricted to completed/defined target records. Censored records are excluded from bad-rate denominators.


In [ ]:
if len(completed):
    completed_summary = pd.DataFrame([
        {
            "completed_rows": len(completed),
            "good_rows": int((completed["target_bad"] == 0).sum()),
            "bad_rows": int((completed["target_bad"] == 1).sum()),
            "observed_bad_rate": round(completed["target_bad"].mean(), 4),
            "share_of_working_sample_completed": round(len(completed) / len(accepted), 4),
        }
    ])
    display_and_save_table(completed_summary, "accepted_completed_target_summary")


In [ ]:
for col in ["grade", "sub_grade", "term", "purpose", "home_ownership", "verification_status", "application_type", "addr_state"]:
    if col not in completed:
        continue
    br = bad_rate_by_group(completed, col, min_n=CONFIG.min_group_n)
    display_and_save_table(br, f"bad_rate_by_{col}")

    if br.empty:
        continue
    plot_br = br.sort_values("bad_rate", ascending=True).tail(25)
    plt.figure(figsize=(10, max(4, 0.28 * len(plot_br))))
    sns.barplot(data=plot_br, x="bad_rate", y=col, color="#D55E00")
    plt.title(f"Observed Bad Rate by {col}")
    plt.xlabel("Bad rate among completed loans")
    plt.ylabel("")
    save_current_plot(f"observed_bad_rate_by_{col}")
    plt.show()


In [ ]:
def add_quantile_band(df: pd.DataFrame, col: str, q: int = 10) -> pd.Series:
    s = pd.to_numeric(df[col], errors="coerce")
    try:
        return pd.qcut(s, q=q, duplicates="drop")
    except ValueError:
        return pd.Series(pd.NA, index=df.index, dtype="object")

banded = completed.copy()
for col in ["fico_mean", "dti", "annual_inc", "loan_amnt", "int_rate_clean", "revol_util_clean", "credit_history_years"]:
    if col in banded:
        band_col = f"{col}_band"
        banded[band_col] = add_quantile_band(banded, col, q=10)
        br = bad_rate_by_group(banded, band_col, min_n=CONFIG.min_group_n)
        display_and_save_table(br, f"bad_rate_by_{band_col}")
        if not br.empty:
            plot_br = br.copy()
            plot_br[band_col] = plot_br[band_col].astype(str)
            plt.figure(figsize=(11, 4))
            sns.lineplot(data=plot_br, x=band_col, y="bad_rate", marker="o", sort=False)
            plt.xticks(rotation=45, ha="right")
            plt.title(f"Observed Bad Rate by {col} Decile Band")
            plt.xlabel(col)
            plt.ylabel("Bad rate among completed loans")
            save_current_plot(f"observed_bad_rate_by_{band_col}")
            plt.show()


In [ ]:
if {"issue_year", "target_bad"}.issubset(completed.columns):
    vintage = (
        completed.groupby("issue_year")
        .agg(n=("target_bad", "count"), bad_rate=("target_bad", "mean"), bad_count=("target_bad", "sum"))
        .reset_index()
    )
    vintage = vintage[vintage["n"] >= CONFIG.min_group_n]
    vintage["bad_rate"] = vintage["bad_rate"].round(4)
    vintage["bad_count"] = vintage["bad_count"].astype(int)
    display_and_save_table(vintage, "accepted_vintage_bad_rate_by_issue_year")

    plt.figure(figsize=(11, 5))
    sns.lineplot(data=vintage, x="issue_year", y="bad_rate", marker="o", color="#D55E00")
    plt.title("Completed Accepted Loans: Observed Bad Rate by Issue Year")
    plt.xlabel("Issue year")
    plt.ylabel("Observed bad rate")
    save_current_plot("accepted_vintage_bad_rate_by_issue_year")
    plt.show()


## 11. Relationships Among Risk, Pricing, and Loan Structure

These charts check whether underwriting grade, interest rate, FICO, DTI, amount, and term move in economically coherent directions. This is EDA, not causal inference.


In [ ]:
if {"grade", "int_rate_clean"}.issubset(accepted.columns):
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=accepted, x="grade", y="int_rate_clean", order=sorted(accepted["grade"].dropna().unique()))
    plt.title("Interest Rate by LendingClub Grade")
    plt.xlabel("Grade")
    plt.ylabel("Interest rate")
    save_current_plot("accepted_interest_rate_by_grade")
    plt.show()

if {"grade", "fico_mean"}.issubset(accepted.columns):
    plt.figure(figsize=(10, 5))
    sns.boxplot(data=accepted, x="grade", y="fico_mean", order=sorted(accepted["grade"].dropna().unique()))
    plt.title("FICO Mean by LendingClub Grade")
    plt.xlabel("Grade")
    plt.ylabel("FICO mean")
    save_current_plot("accepted_fico_by_grade")
    plt.show()


In [ ]:
corr_cols = [
    "loan_amnt", "int_rate_clean", "installment", "annual_inc", "dti", "fico_mean",
    "emp_length_years", "credit_history_years", "open_acc", "revol_bal", "revol_util_clean",
    "total_acc", "mort_acc", "bc_util", "tot_cur_bal", "target_bad",
]
corr_cols = [c for c in corr_cols if c in completed]
if len(corr_cols) >= 3:
    corr = completed[corr_cols].corr(numeric_only=True)
    display_and_save_table(corr.reset_index(names="column"), "accepted_completed_numeric_correlation")

    plt.figure(figsize=(12, 9))
    sns.heatmap(corr, cmap="vlag", center=0, linewidths=0.2)
    plt.title("Accepted Completed Loans: Numeric Correlations")
    save_current_plot("accepted_completed_numeric_correlation_heatmap")
    plt.show()


## 12. Leakage Audit

Accepted-loan files contain many fields created after origination: payments, outstanding principal, recoveries, hardship, settlement, last FICO, and last credit-pull fields. These are useful for servicing analysis but invalid for an at-origination repayment-risk model.


In [ ]:
CLEAR_LEAKAGE_EXCLUDE = [
    "pymnt_plan", "out_prncp", "out_prncp_inv", "total_pymnt", "total_pymnt_inv",
    "total_rec_prncp", "total_rec_int", "total_rec_late_fee", "recoveries",
    "collection_recovery_fee", "last_pymnt_d", "last_pymnt_amnt", "next_pymnt_d",
    "last_credit_pull_d", "last_fico_range_high", "last_fico_range_low", "hardship_flag",
    "hardship_type", "hardship_reason", "hardship_status", "deferral_term", "hardship_amount",
    "hardship_start_date", "hardship_end_date", "payment_plan_start_date", "hardship_length",
    "hardship_dpd", "hardship_loan_status", "orig_projected_additional_accrued_interest",
    "hardship_payoff_balance_amount", "hardship_last_payment_amount", "debt_settlement_flag",
    "debt_settlement_flag_date", "settlement_status", "settlement_date", "settlement_amount",
    "settlement_percentage", "settlement_term",
]

AMBIGUOUS_REVIEW_REQUIRED = [
    "funded_amnt", "funded_amnt_inv", "url", "desc", "title", "zip_code",
    "initial_list_status", "policy_code", "disbursement_method", "emp_title", "addr_state",
]

MODEL_EXCLUDE_IDENTIFIER_TARGET = ["id", "member_id", "loan_status", "target_bad", "target_definition"]

leakage_audit = []
for col in accepted_columns:
    if col in CLEAR_LEAKAGE_EXCLUDE:
        label = "exclude_clear_post_origination_leakage"
    elif col in AMBIGUOUS_REVIEW_REQUIRED:
        label = "review_timing_or_compliance_before_use"
    elif col in MODEL_EXCLUDE_IDENTIFIER_TARGET:
        label = "exclude_identifier_or_target"
    elif col in APPLICATION_TIME_CANDIDATES:
        label = "candidate_application_time_feature"
    else:
        label = "not_in_working_sample_review_before_use"
    leakage_audit.append({"column": col, "recommendation": label})

leakage_audit = pd.DataFrame(leakage_audit)
display_and_save_table(leakage_audit, "accepted_leakage_audit_all_raw_columns")
leakage_audit["recommendation"].value_counts()


## 13. Modeling-Readiness Recommendations

Recommended supervised modeling frame:

- Population: accepted/originated loans only.
- Label: completed-loan `target_bad`, where bad is charged-off/default/late and good is fully paid.
- Exclude: current/unresolved loans from binary PD training, unless using survival analysis.
- Split: chronological train/validation/test by `issue_d`, not random-only validation.
- Features: application-time variables after leakage, timing, missingness, and compliance review.
- Geography/text: treat `addr_state`, `zip_code`, `title`, `desc`, and `emp_title` as proxy-risk or privacy-risk fields requiring fairness and governance review.
- Calibration: evaluate rank ordering and probability calibration by vintage, grade, term, FICO, DTI, and loan amount.

Key limitation: because denied loans are intentionally excluded, this model estimates repayment risk **conditional on approval**. It does not estimate default risk for rejected applicants or optimize approval policy across all applicants.


In [ ]:
SAFE_STARTER_FEATURES = [
    "loan_amnt", "term_months", "int_rate_clean", "installment", "grade", "sub_grade",
    "emp_length_years", "home_ownership", "annual_inc", "verification_status", "purpose",
    "dti", "delinq_2yrs", "fico_mean", "inq_last_6mths", "mths_since_last_delinq",
    "mths_since_last_record", "open_acc", "pub_rec", "revol_bal", "revol_util_clean",
    "total_acc", "credit_history_years", "collections_12_mths_ex_med", "acc_now_delinq",
    "tot_coll_amt", "tot_cur_bal", "acc_open_past_24mths", "avg_cur_bal", "bc_open_to_buy",
    "bc_util", "mort_acc", "pub_rec_bankruptcies", "tax_liens", "total_bal_ex_mort",
    "total_bc_limit", "total_il_high_credit_limit", "application_type",
]
SAFE_STARTER_FEATURES = [c for c in SAFE_STARTER_FEATURES if c in accepted.columns]

modeling_readiness = pd.DataFrame([
    {"item": "source_population", "recommendation": "Accepted/originated LendingClub loans only"},
    {"item": "target", "recommendation": "target_bad on completed loans; exclude unresolved current loans for binary PD"},
    {"item": "validation", "recommendation": "Use issue-date chronological split; report vintage stability"},
    {"item": "leakage", "recommendation": "Drop all payment, recovery, hardship, settlement, last-FICO, and last-credit-pull fields"},
    {"item": "fair_lending", "recommendation": "Review geography, text, employment title, and any proxy-risk variables before production framing"},
    {"item": "starter_feature_count", "recommendation": str(len(SAFE_STARTER_FEATURES))},
])
display_and_save_table(modeling_readiness, "accepted_modeling_readiness_recommendations")

pd.Series(SAFE_STARTER_FEATURES, name="starter_feature").to_csv(TABLE_DIR / "accepted_safe_starter_features.csv", index=False)
print("Starter feature list saved:", TABLE_DIR / "accepted_safe_starter_features.csv")
print(SAFE_STARTER_FEATURES)


In [ ]:
if "issue_d_dt" in completed and len(completed):
    split_frame = completed[["issue_d_dt", "target_bad"]].dropna().sort_values("issue_d_dt")
    q_train, q_valid = split_frame["issue_d_dt"].quantile([0.70, 0.85])
    split_plan = pd.DataFrame([
        {"split": "train", "date_rule": f"issue_d <= {q_train.date()}"},
        {"split": "validation", "date_rule": f"{q_train.date()} < issue_d <= {q_valid.date()}"},
        {"split": "test", "date_rule": f"issue_d > {q_valid.date()}"},
    ])
    display_and_save_table(split_plan, "accepted_recommended_time_split_plan")


## 14. Final EDA Conclusions

After running this notebook, use the generated tables and plots under `accepted_eda_outputs/` to support the final modeling plan. The most important governance conclusions should remain stable even if the sample size is changed:

1. The accepted-loan file contains valid repayment outcomes; denied loans are not needed for accepted-population repayment-risk modeling.
2. `loan_status` must be converted into a carefully documented target, and unresolved loans should not be treated as good loans.
3. Time drift is expected; validation should be chronological and vintage-aware.
4. Many accepted-loan fields are post-origination leakage and must be excluded from at-origination models.
5. Geography and text fields require fair-lending/privacy review before use.
6. Any final model should document that it predicts repayment risk among borrowers who passed LendingClub's historical acceptance process.
